# Phase 10 — SQL Validation & Security

This notebook validates LLM-generated SQL before execution.

Only read-only SELECT statements are allowed.

Blocked operations include:

- DROP
- DELETE
- UPDATE
- INSERT
- ALTER
- CREATE
- TRUNCATE
- GRANT
- REVOKE
- COPY INTO
- filesystem access
- multiple SQL statements
- unauthorized tables

The validator returns a standard response:

{
    "valid": True/False,
    "normalized_sql": "...",
    "reason": "..."
}

In [0]:
import re

print("SQL Validator initialized.")

In [0]:
ALLOWED_TABLES = {
    "genai_copilot.gold.monthly_sales",
    "genai_copilot.gold.region_sales",
    "genai_copilot.gold.customer_metrics",
    "genai_copilot.gold.product_metrics",
    "genai_copilot.gold.category_metrics",
}

MAX_SQL_LENGTH = 10000

print("Allowed tables:")
for table in sorted(ALLOWED_TABLES):
    print(" -", table)

print("\nMaximum SQL length:", MAX_SQL_LENGTH)

In [0]:
BLOCKED_KEYWORDS = [
    "DROP",
    "DELETE",
    "UPDATE",
    "INSERT",
    "ALTER",
    "CREATE",
    "TRUNCATE",
    "GRANT",
    "REVOKE",
    "COPY",
    "MERGE",
    "REPLACE",
]

print("Blocked SQL operations:")
print(BLOCKED_KEYWORDS)

In [0]:
def normalize_sql(sql):
    """
    Normalize SQL for validation.
    """

    if not isinstance(sql, str):
        raise ValueError("SQL must be a string.")

    sql = sql.strip()

    if not sql:
        raise ValueError("SQL cannot be empty.")

    return sql

In [0]:
def contains_multiple_statements(sql):
    """
    Detect multiple SQL statements.

    A semicolon is allowed only as the final character.
    """

    sql_without_trailing_semicolon = sql.rstrip().rstrip(";")

    return ";" in sql_without_trailing_semicolon

In [0]:
def find_blocked_keyword(sql):
    """
    Return the first dangerous SQL keyword found.
    """

    for keyword in BLOCKED_KEYWORDS:

        pattern = rf"\b{keyword}\b"

        if re.search(
            pattern,
            sql,
            flags=re.IGNORECASE
        ):
            return keyword

    return None

In [0]:
def extract_tables(sql):
    """
    Extract tables appearing after FROM or JOIN.
    """

    pattern = r"""
        \b(?:FROM|JOIN)\s+
        ([A-Za-z_][\w]*\.[A-Za-z_][\w]*\.[A-Za-z_][\w]*)
    """

    matches = re.findall(
        pattern,
        sql,
        flags=re.IGNORECASE | re.VERBOSE
    )

    return set(
        table.lower()
        for table in matches
    )

In [0]:
def validate_sql(sql):
    """
    Validate SQL before execution.

    Returns a consistent dictionary:

    {
        "valid": True/False,
        "normalized_sql": "...",
        "reason": "..."
    }
    """

    # -----------------------------------------------
    # Basic validation
    # -----------------------------------------------

    try:
        normalized_sql = normalize_sql(sql)

    except Exception as e:

        return {
            "valid": False,
            "normalized_sql": "",
            "reason": str(e)
        }

    # -----------------------------------------------
    # Length validation
    # -----------------------------------------------

    if len(normalized_sql) > MAX_SQL_LENGTH:

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": (
                f"SQL exceeds maximum allowed length "
                f"of {MAX_SQL_LENGTH} characters."
            )
        }

    # -----------------------------------------------
    # Remove final semicolon
    # -----------------------------------------------

    normalized_sql = normalized_sql.strip()

    if normalized_sql.endswith(";"):
        normalized_sql = normalized_sql[:-1].strip()

    # -----------------------------------------------
    # Multiple statements
    # -----------------------------------------------

    if contains_multiple_statements(
        normalized_sql
    ):

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": "Multiple SQL statements are not allowed."
        }

    # -----------------------------------------------
    # Must begin with SELECT or WITH
    # -----------------------------------------------

    if not re.match(
        r"^(SELECT|WITH)\b",
        normalized_sql,
        flags=re.IGNORECASE
    ):

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": (
                "Only SELECT or WITH queries are allowed."
            )
        }

    # -----------------------------------------------
    # Dangerous keywords
    # -----------------------------------------------

    blocked_keyword = find_blocked_keyword(
        normalized_sql
    )

    if blocked_keyword:

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": (
                f"Blocked SQL operation detected: "
                f"{blocked_keyword}"
            )
        }

    # -----------------------------------------------
    # Filesystem / external access
    # -----------------------------------------------

    external_patterns = [
        r"\bREAD_FILES\b",
        r"\bINPUT_FILE_NAME\b",
        r"\bDBFS\b",
        r"\bS3A?://",
        r"\bABFSS?://",
        r"\bWASBS?://",
        r"\bHTTP://",
        r"\bHTTPS://",
    ]

    for pattern in external_patterns:

        if re.search(
            pattern,
            normalized_sql,
            flags=re.IGNORECASE
        ):

            return {
                "valid": False,
                "normalized_sql": normalized_sql,
                "reason": (
                    "Filesystem or external data access "
                    "is not allowed."
                )
            }

    # -----------------------------------------------
    # Extract tables
    # -----------------------------------------------

    referenced_tables = extract_tables(
        normalized_sql
    )

    # -----------------------------------------------
    # Require at least one approved table
    # -----------------------------------------------

    if not referenced_tables:

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": (
                "No approved Gold table was detected."
            )
        }

    # -----------------------------------------------
    # Table allowlist
    # -----------------------------------------------

    unauthorized_tables = (
        referenced_tables - {
            table.lower()
            for table in ALLOWED_TABLES
        }
    )

    if unauthorized_tables:

        return {
            "valid": False,
            "normalized_sql": normalized_sql,
            "reason": (
                "Unauthorized table(s): "
                + ", ".join(
                    sorted(unauthorized_tables)
                )
            )
        }

    # -----------------------------------------------
    # Success
    # -----------------------------------------------

    return {
        "valid": True,
        "normalized_sql": normalized_sql,
        "reason": "SQL validation successful."
    }

In [0]:
test_sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
"""

validation_result = validate_sql(test_sql)

print(validation_result)


10.35 Security layers in our project

Our final system will have multiple security layers:

Layer 1 — Prompt

Don't send unnecessary sensitive information to the LLM.

Layer 2 — Schema allowlist

Only expose relevant schemas/tables.

Layer 3 — SQL validator

Reject dangerous SQL.

Layer 4 — Table allowlist

Only Gold analytical tables.

Layer 5 — Read-only execution

Only approved SELECT statements.

Layer 6 — Result limits

Prevent unnecessarily large responses.

Layer 7 — Databricks permissions

Unity Catalog permissions remain the final authorization layer.

The SQL Statement Execution API also enforces permissions on objects used by the statement.